# Modulo 1 · Notebook 3 — Pipeline de Entrenamiento

En el Notebook 2 definimos features en **Feast** y aprendimos a materializarlas y
servirlas. Pero un feature store no es un fin en si mismo: existe para
**alimentar modelos**. En este notebook **cerramos el ciclo**:

```
   feature store  ->  set de entrenamiento  ->  modelo  ->  MLflow
```

Es decir:

1. Construimos el **set de entrenamiento** leyendo del **offline store** de Feast
   con `get_historical_features` (con un fallback a parquet si Feast no esta
   disponible).
2. Entrenamos un **clasificador simple** que predice `survived`.
3. Lo **evaluamos** (accuracy y ROC-AUC).
4. Registramos parametros, metricas y el modelo en **MLflow**.

> **La idea grande: consistencia entrenamiento-serving.** El set de entrenamiento
> sale de las *mismas* definiciones de features que luego serviran online. Por eso
> el modelo ve en produccion exactamente la misma representacion con la que
> aprendio: no hay sorpresas de *skew*. El feature store es el contrato que une
> ambos mundos.

MLflow es el **servidor de tracking compartido del curso** (lo introduce este
modulo y lo usan todos): desde el host esta en `http://localhost:5000`; dentro de
la red de Docker, en `http://mlflow:5000`.


## Setup

Definimos las rutas y un helper para localizar el repo de Feast y el parquet del
offline store. No se necesita ningun servicio levantado para la ruta de parquet;
para usar Feast directamente, primero corre `feast apply` (Notebook 2).

In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))
PARQUET_PATH = os.path.join(REPO, "data", "titanic_features.parquet")
print("repo de features:", REPO)
print("parquet offline :", PARQUET_PATH)
print("existe parquet  :", os.path.exists(PARQUET_PATH))

# URI de tracking de MLflow: localhost desde el host, mlflow:5000 dentro de Docker.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
print("MLflow tracking :", MLFLOW_TRACKING_URI)

## Paso 1 — construir el set de entrenamiento desde el feature store

La forma *correcta* de armar un set de entrenamiento es con
`get_historical_features`: le pasas un **dataframe de entidades** (que pasajeros y
en que `event_timestamp`) y te devuelve las features validas en ese momento
(point-in-time). Aqui usamos como timestamp "ahora", asi que traemos el ultimo
valor de cada pasajero.

Si Feast no esta instalado o el registry no esta aplicado, caemos a **leer el
parquet del offline store directamente** (la misma fuente que Feast usa por
debajo). En ambos casos obtenemos las mismas features.

In [ ]:
def load_offline_parquet():
    """Fallback: lee directamente el parquet del offline store de Feast."""
    df = pd.read_parquet(PARQUET_PATH)
    print(f"[parquet] {len(df)} filas leidas de {PARQUET_PATH}")
    return df

FEATURES = [
    "titanic_passenger_features:age_scaled",
    "titanic_passenger_features:fare_log",
    "titanic_passenger_features:family_size",
    "titanic_passenger_features:is_alone",
    "titanic_passenger_features:pclass",
    "titanic_passenger_features:sex",
    "titanic_passenger_features:embark_town",
]

def build_training_set():
    """Intenta Feast (get_historical_features); si falla, usa el parquet."""
    try:
        from feast import FeatureStore
        store = FeatureStore(repo_path=REPO)

        base = pd.read_parquet(PARQUET_PATH)  # de aqui sacamos entidades + etiqueta
        entity_df = pd.DataFrame({
            "passenger_id": base["passenger_id"].values,
            "event_timestamp": [datetime.utcnow()] * len(base),
        })
        feats = store.get_historical_features(
            entity_df=entity_df, features=FEATURES,
        ).to_df()
        # Adjuntamos la etiqueta `survived` desde el parquet (no es una feature de Feast).
        feats = feats.merge(base[["passenger_id", "survived"]], on="passenger_id")
        print(f"[feast] set de entrenamiento point-in-time: {feats.shape}")
        return feats
    except Exception as e:
        print(f"[feast] no disponible ({type(e).__name__}: {e}); uso el parquet.")
        return load_offline_parquet()

train_df = build_training_set()
train_df.head()

## Paso 2 — entrenar un clasificador simple

Predecimos `survived` (sobrevivio o no) con una **regresion logistica** dentro de
un `Pipeline` de sklearn. El pipeline encapsula el preprocesamiento (one-hot de
las categoricas) junto con el modelo, de modo que *exactamente los mismos pasos*
corren en entrenamiento y en serving. Eso es, de nuevo, consistencia
entrenamiento-serving, ahora a nivel de codigo del modelo.

Las features numericas (`age_scaled`, `fare_log`, ...) ya vienen procesadas desde
el feature store; solo nos falta codificar las categoricas `sex` y
`embark_town`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric = ["age_scaled", "fare_log", "family_size", "is_alone", "pclass"]
categorical = ["sex", "embark_town"]

# Solo conservamos columnas que existan (robusto al fallback de parquet).
numeric = [c for c in numeric if c in train_df.columns]
categorical = [c for c in categorical if c in train_df.columns]

X = train_df[numeric + categorical].copy()
y = train_df["survived"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough",
)
model = Pipeline(steps=[
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
print("Modelo entrenado. Features:", numeric + categorical)

## Paso 3 — evaluar

Medimos **accuracy** (fraccion de aciertos) y **ROC-AUC** (que tan bien el modelo
ordena positivos por encima de negativos, robusto al desbalance de clases).

In [ ]:
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

try:
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
except Exception:
    auc = float("nan")

print(f"accuracy : {acc:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

## Paso 4 — registrar en MLflow

Registramos los **parametros** (tipo de modelo, tamanos de split), las
**metricas** (accuracy, ROC-AUC) y el **modelo** mismo en MLflow. A partir de aqui
el run queda versionado y comparable con el resto de experimentos del curso.

Si el servidor de MLflow no esta levantado (`docker compose up -d mlflow` en
`../platform`), MLflow cae a un backend local en `./mlruns`, asi que la celda no
rompe el notebook.

In [ ]:
import mlflow
import mlflow.sklearn

try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("module1_feature_pipeline")
    target_uri = MLFLOW_TRACKING_URI
except Exception as e:
    print(f"Servidor MLflow no alcanzable ({e}); uso backend local ./mlruns")
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("module1_feature_pipeline")
    target_uri = "file:./mlruns"

with mlflow.start_run(run_name="notebook03_train") as run:
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_metric("accuracy", float(acc))
    if auc == auc:  # no es NaN
        mlflow.log_metric("roc_auc", float(auc))
    mlflow.sklearn.log_model(model, artifact_path="model")
    run_id = run.info.run_id

print(f"Run registrado: {run_id}")
print(f"Tracking en: {target_uri}")
print("Abre http://localhost:5000 para ver el experimento (si el servidor corre).")

## Repaso — el ciclo cerrado

Acabamos de recorrer el ciclo completo de un sistema de ML del mundo real:

1. **Feature store (Feast)** define y sirve features de forma consistente.
2. `get_historical_features` arma un **set de entrenamiento** point-in-time
   correcto (sin leakage).
3. Un **modelo** se entrena con ese set, dentro de un pipeline que garantiza que
   el preprocesamiento sera identico en serving.
4. **MLflow** registra el experimento (params, metricas, modelo) para que sea
   reproducible y comparable.

**Por que importa la consistencia entrenamiento-serving:** el modelo aprende sobre
una representacion concreta de los datos. Si en produccion las features se
calculan de otra forma (otra mediana de imputacion, otro escalado, otra version de
la transformacion), el modelo recibe entradas "fuera de distribucion" y se degrada
en silencio. El feature store + el pipeline de sklearn son las dos piezas que
garantizan que *lo que se entrena es lo que se sirve*.

**Orquestacion.** Este mismo ciclo lo automatiza el DAG de **Apache Airflow** de la
plataforma compartida (`platform/dags/feature_engineering_dag.py`):
`extract -> prepare -> transform -> validate -> feast_apply -> feast_materialize ->
train_model`. Airflow es el **orquestador compartido del curso**, introducido en
este modulo.